# Train the 16 MB Bangla speech model

The network is `model/student.py`. It is built at random weights and is checked to stay under **16.9 MB** at 1 byte per weight. That is the Whistle-sized file. Whisper tiny is about 75 MB, so it is not this model.

Quality at that size comes from two teachers of the loss, not from a bigger file:

- Human Bangla transcripts (Common Voice train split).
- A frozen `openai/whisper-small` encoder. Its vectors supervise our encoder. Whisper itself is not saved.

Kaggle: GPU T4, Internet On. Add this repo folder as Input data if `model/student.py` is not beside the notebook. Set `SMOKE = False` for the paper run.

If the GPU runs out of memory, change `TEACHER_ID` to `openai/whisper-base` and run again. Small is the better teacher when it fits.


In [ ]:
SMOKE = False
TEACHER_ID = "openai/whisper-small"
OUT_DIR = "/kaggle/working/whistle-bn"
DEVICE = "cuda"
N_ZIPS = 1 if SMOKE else 8  # zips 0..7 train. Never train on asr_bengali_f, it is the local test set
MAX_STEPS = 40 if SMOKE else 20000  # 4000 steps at batch 1 (first run) never learned: CER 0.90
TRAIN_ROWS = 64 if SMOKE else 10**9
VAL_ROWS = 8 if SMOKE else 300
BATCH = 4 if SMOKE else 32
LR = 1e-3
WARMUP = 10 if SMOKE else 1000
KD_WEIGHT = 1.0  # 0 skips the teacher
EVAL_EVERY = 20 if SMOKE else 2000

In [ ]:
import torch
print("cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Turn on Kaggle GPU T4. Do not train on CPU.")
print("GPU", torch.cuda.get_device_name(0))


In [ ]:
%pip install -q transformers datasets sentencepiece accelerate librosa soundfile


In [ ]:
import sys
from pathlib import Path

found = list(Path("/kaggle/input").rglob("model/student.py")) if Path("/kaggle/input").exists() else []
found += list(Path.cwd().rglob("model/student.py"))
if not found:
    raise FileNotFoundError("model/student.py not found under /kaggle/input")
root = found[0].parents[1]
print("using", root)
sys.path.insert(0, str(root))
from model.student import StudentASR, file_bytes

student = StudentASR()
print(f"int8 file {file_bytes(student) / 1e6:.2f} MB (budget 16.9)")


### Data and the Bangla syllable tokenizer

[OpenSLR 53](https://www.openslr.org/53/), no token needed. The full run downloads zips 0–7 (about 7 GB, each deleted after unzip). Zip `f` is never used for training; it is the local held-out check. 300 rows are held out for validation CER.


In [ ]:
import random
import urllib.request
import zipfile
from pathlib import Path

base = Path("/kaggle/temp/openslr53")  # not /kaggle/working, or the output zip is the audio
wav_root = base / "wavs"
base.mkdir(parents=True, exist_ok=True)
wav_root.mkdir(exist_ok=True)
tsv_path = base / "utt_spk_text.tsv"
if not tsv_path.exists():
    urllib.request.urlretrieve("https://www.openslr.org/resources/53/utt_spk_text.tsv", tsv_path)

for ch in "0123456789abcdef"[:N_ZIPS]:
    zpath = base / f"asr_bengali_{ch}.zip"
    marker = base / f".got_{ch}"
    if not marker.exists():
        print("downloading", zpath.name)
        urllib.request.urlretrieve(f"https://www.openslr.org/resources/53/{zpath.name}", zpath)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(wav_root)
        zpath.unlink()  # /kaggle/working is 20 GB, zip plus flacs would not fit
        marker.write_text("ok")

clips = {p.stem: str(p) for p in wav_root.rglob("*") if p.suffix.lower() in {".flac", ".wav"}}
rows = []
with tsv_path.open(encoding="utf-8") as handle:
    for line in handle:
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 3:
            continue
        hit = clips.get(Path(parts[0]).stem)
        if hit is not None:
            rows.append((hit, parts[-1]))
if not rows:
    sample = [str(p.relative_to(wav_root)) for p in list(wav_root.rglob("*"))[:8]]
    raise RuntimeError(f"no clips matched the tsv. audio files {len(clips)}. sample {sample}")
random.Random(0).shuffle(rows)
val_rows, train_rows = rows[:VAL_ROWS], rows[VAL_ROWS:VAL_ROWS + TRAIN_ROWS]
print("train", len(train_rows), "val", len(val_rows), "clips_on_disk", len(clips))

In [ ]:
from model.bangla_tokenizer import BanglaSyllableTokenizer

sp = BanglaSyllableTokenizer.train([t for _, t in train_rows], vocab_size=4096)
sp.save("/tmp/bn.syllables.json")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
import shutil
shutil.copy("/tmp/bn.syllables.json", Path(OUT_DIR) / "bn.syllables.json")
sample = train_rows[0][1]
print("tokenizer bangla-syllable vocab", sp.vocab_size())
print(sample)
print(sp.decode(sp.encode(sample)))

### Frozen teacher

Whisper-small stays frozen. We only copy its encoder vectors into the small model.


In [ ]:
import torch.nn as nn
from transformers import WhisperForConditionalGeneration, WhisperFeatureExtractor

feat = WhisperFeatureExtractor.from_pretrained(TEACHER_ID)
teacher = None
if KD_WEIGHT:
    teacher = WhisperForConditionalGeneration.from_pretrained(TEACHER_ID, torch_dtype=torch.float16).model.encoder
    teacher.to(DEVICE).eval().requires_grad_(False)
proj = nn.Linear(256, 768 if teacher is None else teacher.config.d_model).to(DEVICE)
student.to(DEVICE)

In [ ]:
import time
import soundfile as sf
import torch.nn.functional as F
from torch.utils.data import DataLoader

from model.student import mel_frames


def collate(batch):
    audio = [sf.read(path, dtype="float32")[0][: 16000 * 30] for path, _ in batch]
    # Whisper features for 30 s; the student only sees the real frames, the teacher needs all 3000.
    mel = feat(audio, sampling_rate=16000, return_tensors="pt", padding="max_length").input_features
    frames = torch.tensor([mel_frames(len(a)) for a in audio])
    ids = [[sp.bos_id()] + sp.encode(text)[:254] + [sp.eos_id()] for _, text in batch]
    labels = torch.full((len(ids), max(map(len, ids))), sp.pad_id(), dtype=torch.long)
    for i, piece in enumerate(ids):
        labels[i, : len(piece)] = torch.tensor(piece)
    return mel, frames, labels


def edit_distance(a, b):
    d = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, y in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (x != y))
    return d[-1]


def val_cer(rows_):
    student.eval()
    errors = chars = 0
    for mel, frames, labels in DataLoader(rows_, batch_size=1, collate_fn=collate):
        ids = student.generate(mel[..., : int(frames[0])].to(DEVICE), sp.bos_id(), sp.eos_id())[0].tolist()
        ref = sp.decode(labels[0].tolist()).replace(" ", "")
        errors += edit_distance(ref, sp.decode(ids).replace(" ", ""))
        chars += len(ref)
    student.train()
    return errors / max(chars, 1)


loader = DataLoader(train_rows, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=4, drop_last=True, persistent_workers=True)
opt = torch.optim.AdamW(list(student.parameters()) + list(proj.parameters()), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / WARMUP, max(0.0, (MAX_STEPS - s) / (MAX_STEPS - WARMUP))))
amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler(enabled=amp)
out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
student.train()
step, best, t0 = 0, 9.9, time.time()
while step < MAX_STEPS:
    for mel, frames, labels in loader:
        mel, frames, labels = mel.to(DEVICE), frames.to(DEVICE), labels.to(DEVICE)
        t = int(frames.max())
        pad = torch.arange(t, device=DEVICE)[None] >= frames[:, None]
        enc_pad = pad[:, ::2]
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            memory = student.encode(mel[..., :t], pad)
            logits = student.decode(memory, labels[:, :-1], mem_pad=enc_pad)
            loss_ce = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1), ignore_index=sp.pad_id())
            loss_kd = torch.zeros((), device=DEVICE)
            if teacher is not None:
                with torch.no_grad():
                    teacher_h = teacher(mel.half()).last_hidden_state.float()
                diff = (proj(memory).float() - teacher_h[:, : memory.size(1)]).pow(2).mean(-1)
                loss_kd = diff[~enc_pad].mean()  # real frames only, not 30 s of padding
            loss = loss_ce + KD_WEIGHT * loss_kd
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(list(student.parameters()) + list(proj.parameters()), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        step += 1
        if step % 100 == 0 or step == 1 or SMOKE:
            print(f"step {step} ce {loss_ce.item():.3f} kd {loss_kd.item():.3f} lr {sched.get_last_lr()[0]:.2e} {time.time() - t0:.0f}s", flush=True)
        if step % EVAL_EVERY == 0 or step == MAX_STEPS:
            cer = val_cer(val_rows)
            print(f"step {step} val CER {cer:.3f}", flush=True)
            if cer < best:
                best = cer
                torch.save(student.state_dict(), out / "student.pt")  # keeps the best so far if Kaggle times out
        if step >= MAX_STEPS:
            break
print("best val CER", best)

In [ ]:
import shutil
from model.student import pack_int8

feat.save_pretrained(out)
shutil.copy("/tmp/bn.syllables.json", out / "bn.syllables.json")
torch.save(pack_int8(torch.load(out / "student.pt", map_location="cpu")), out / "student.int8.pt")
print("saved", out, "student.int8.pt MB", round((out / "student.int8.pt").stat().st_size / 1e6, 2))